# Argentina SYNOP cloud base height retrieval

Starter notebook for **cloudVis**: download surface synoptic (SYNOP) observations over Argentina and extract **cloud base height**.

**Target variable**
- **Cloud base height** from SYNOP **Section 3**, group **`8NChh`**
- `hh` is interpreted as **hundreds of feet** → `cloud_base_ft = hh × 100`
- `cloud_base_m` is derived from feet for convenience

**Data sources**
1. [OGIMET raw SYNOP API](http://www.ogimet.com/cgi-bin/getsynop) — primary source for Section 3 cloud layers
2. [`atmofetch`](https://pypi.org/project/atmofetch/) / OGIMET decoded hourly tables — optional fallback when raw data is unavailable

Later steps: spatial interpolation over Argentina and optional export to AERMET-compatible formats.

In [ ]:
# Run this cell first in Google Colab
!pip install -q atmofetch pandas requests folium

In [ ]:
from __future__ import annotations

from datetime import date, datetime, timedelta, timezone
from io import StringIO
from typing import Iterable
import re

import pandas as pd
import requests
from atmofetch import meteo_ogimet
from atmofetch._utils.network import fetch_ogimet

# --- user configuration ---
END_UTC = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
START_UTC = END_UTC - timedelta(hours=6)  # keep short for Colab; increase once it works

M_TO_FEET = 3.28084
OGIMET_URL = "http://www.ogimet.com/cgi-bin/getsynop"


def _dms_to_decimal(deg: str, minutes: str, seconds: str | None, hemisphere: str) -> float:
    seconds = seconds or "0"
    value = int(deg) + int(minutes) / 60 + int(seconds) / 3600
    return -value if hemisphere in {"S", "W"} else value


def fetch_stations_argentina(
    country: str = "Argentina",
    date_query: date | None = None,
) -> pd.DataFrame:
    """Fetch SYNOP station coordinates from OGIMET with correct DMS parsing.

    atmofetch.stations_ogimet() mis-parses OGIMET Lat=/Lon= strings and can be
    off by 50-150 km. This parser reads DD-MM-SS (or DD-MM) DMS values directly.
    """
    if date_query is None:
        date_query = date.today()

    url = (
        f"http://ogimet.com/cgi-bin/gsynres?lang=en&state={country.replace(' ', '+')}"
        f"&osum=no&fmt=html&ord=REV&ano={date_query.strftime('%Y')}"
        f"&mes={date_query.strftime('%m')}&day={date_query.strftime('%d')}"
        f"&hora=06&ndays=1&Send=send"
    )
    html = fetch_ogimet(url)

    lat_re = re.compile(r"Lat=(\d+)-(\d+)(?:-(\d+))?([NS])")
    lon_re = re.compile(r"Lon=(\d+)-(\d+)(?:-(\d+))?([EW])")
    alt_re = re.compile(r"Alt=(\d+)")
    meta_re = re.compile(r"'(\d{5}) \(.*? - (.+?)\)'")

    records: list[dict] = []
    for chunk in html.split("Decoded synops since")[1:]:
        lat_m = lat_re.search(chunk)
        lon_m = lon_re.search(chunk)
        meta_m = meta_re.search(chunk)
        if not (lat_m and lon_m and meta_m):
            continue

        alt_m = alt_re.search(chunk)
        records.append(
            {
                "wmo_id": meta_m.group(1),
                "station_names": meta_m.group(2).strip(),
                "lat": _dms_to_decimal(*lat_m.groups()),
                "lon": _dms_to_decimal(*lon_m.groups()),
                "alt": int(alt_m.group(1)) if alt_m else None,
            }
        )

    return pd.DataFrame(records)


def extract_cloud_base_section3(report: str) -> dict | None:
    """Extract cloud base from Section 3 groups 8NChh.

    hh is treated as hundreds of feet (cloud_base_ft = hh * 100).
    When multiple cloud layers are reported, the lowest base is used.
    """
    parts = report.replace("=", " ").split()
    if "333" not in parts:
        return None

    heights_ft: list[float] = []
    hh_codes: list[int] = []

    for group in parts[parts.index("333") + 1 :]:
        if group in {"444", "555"}:
            break
        if len(group) != 5 or not group.startswith("8") or not group[1:].isdigit():
            continue

        cloud_amount = int(group[1])
        hh = int(group[3:5])
        if cloud_amount == 0:
            continue

        heights_ft.append(hh * 100)
        hh_codes.append(hh)

    if not heights_ft:
        return None

    cloud_base_ft = float(min(heights_ft))
    lowest_idx = heights_ft.index(cloud_base_ft)
    cloud_base_hh = hh_codes[lowest_idx]

    return {
        "cloud_base_hh": cloud_base_hh,
        "cloud_base_ft": cloud_base_ft,
        "cloud_base_m": cloud_base_ft / M_TO_FEET,
    }


def download_raw_synop_argentina(start: datetime, end: datetime) -> pd.DataFrame:
    params = {
        "begin": start.strftime("%Y%m%d%H%M"),
        "end": end.strftime("%Y%m%d%H%M"),
        "state": "Argen",
        "header": "yes",
        "lang": "eng",
    }
    response = requests.get(OGIMET_URL, params=params, timeout=120)
    response.raise_for_status()
    raw = pd.read_csv(StringIO(response.text))
    return raw.rename(
        columns={
            "WMO_ID": "station_id",
            "ANO": "year",
            "MES": "month",
            "DIA": "day",
            "HORA": "hour",
            "MINUTO": "minute",
            "PARTE": "report",
        }
    )


def decode_raw_synop_df(raw_df: pd.DataFrame, stations_df: pd.DataFrame) -> pd.DataFrame:
    """Decode raw OGIMET SYNOP bulletins into Section 3 cloud-base observations."""
    records: list[dict] = []
    for _, row in raw_df.iterrows():
        cloud_base = extract_cloud_base_section3(row["report"])
        if cloud_base is None:
            continue

        station_id = str(row["station_id"])
        records.append(
            {
                "station_id": station_id,
                "datetime_utc": pd.Timestamp(
                    year=int(row["year"]),
                    month=int(row["month"]),
                    day=int(row["day"]),
                    hour=int(row["hour"]),
                    minute=int(row["minute"]),
                    tz="UTC",
                ),
                "cloud_base_hh": cloud_base["cloud_base_hh"],
                "cloud_base_ft": cloud_base["cloud_base_ft"],
                "cloud_base_m": cloud_base["cloud_base_m"],
                "report": row["report"],
            }
        )

    if not records:
        return pd.DataFrame()

    decoded_df = pd.DataFrame(records)
    station_lookup = stations_df.rename(columns={"wmo_id": "station_id"}).copy()
    station_lookup["station_id"] = station_lookup["station_id"].astype(str)
    decoded_df["station_id"] = decoded_df["station_id"].astype(str)

    return decoded_df.merge(
        station_lookup[["station_id", "lat", "lon", "station_names"]],
        on="station_id",
        how="left",
    )


def normalize_observations(df: pd.DataFrame) -> pd.DataFrame:
    """Canonical schema for all downstream cells and the map."""
    if df.empty:
        return pd.DataFrame(
            columns=[
                "station_ID",
                "Date",
                "lat",
                "lon",
                "station_names",
                "cloud_base_hh",
                "cloud_base_ft",
                "cloud_base_m",
                "data_source",
            ]
        )

    out = df.copy()

    if "station_ID" not in out.columns:
        if "station_id" in out.columns:
            out["station_ID"] = pd.to_numeric(out["station_id"].astype(str), errors="coerce")
        else:
            out["station_ID"] = pd.NA

    if "Date" not in out.columns:
        out["Date"] = pd.to_datetime(out["datetime_utc"], utc=True, errors="coerce")
    else:
        out["Date"] = pd.to_datetime(out["Date"], utc=True, errors="coerce")

    if "lat" not in out.columns and "Lat" in out.columns:
        out["lat"] = out["Lat"]
    if "lon" not in out.columns and "Lon" in out.columns:
        out["lon"] = out["Lon"]

    if "cloud_base_ft" not in out.columns and "cloud_base_hh" in out.columns:
        out["cloud_base_ft"] = pd.to_numeric(out["cloud_base_hh"], errors="coerce") * 100

    if "cloud_base_m" not in out.columns and "cloud_base_ft" in out.columns:
        out["cloud_base_m"] = pd.to_numeric(out["cloud_base_ft"], errors="coerce") / M_TO_FEET

    if "data_source" not in out.columns:
        out["data_source"] = "unknown"

    return out


def fetch_decoded_synop(
    station_ids: Iterable[int | str],
    start: datetime,
    end: datetime,
    stations_df: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    """Download SYNOP cloud-base data from Section 3 groups 8NChh."""
    station_ids = [str(s) for s in station_ids]

    raw = download_raw_synop_argentina(start, end)
    raw = raw[raw["station_id"].astype(str).isin(station_ids)]
    decoded = decode_raw_synop_df(raw, stations_df)

    if not decoded.empty:
        mask = (decoded["datetime_utc"] >= pd.Timestamp(start)) & (
            decoded["datetime_utc"] <= pd.Timestamp(end)
        )
        decoded = decoded.loc[mask].copy()
        if not decoded.empty:
            decoded["data_source"] = "ogimet_raw_section3"
            return normalize_observations(decoded), "ogimet_raw_section3"

    print("No Section 3 cloud layers found; trying OGIMET decoded hourly tables...")
    df = meteo_ogimet(
        interval="hourly",
        station=[int(s) for s in station_ids],
        date_range=(start.date().isoformat(), end.date().isoformat()),
        coords=True,
    )

    if not df.empty and "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
        mask = (df["Date"] >= pd.Timestamp(start)) & (df["Date"] <= pd.Timestamp(end))
        df = df.loc[mask].copy()
        if not df.empty:
            df["data_source"] = "ogimet_decoded_no_section3"
            return normalize_observations(df), "ogimet_decoded_no_section3"

    return normalize_observations(pd.DataFrame()), "none"

## 1. List Argentina SYNOP stations (with coordinates)

Argentina WMO block is **87xxx** (South America).

Station coordinates are parsed from OGIMET `Lat=DD-MM-SS` / `Lon=DDD-MM-SS` metadata (not from `atmofetch.stations_ogimet()`, which can be tens of km off).

In [ ]:
stations = fetch_stations_argentina()
print(f"{len(stations)} stations found")
stations.head(10)

## 2. Download cloud base height observations

Parses **Section 3** `8NChh` groups from raw OGIMET SYNOP bulletins (`hh` × 100 = feet).
If no Section 3 cloud layers are found, it falls back to OGIMET decoded hourly tables (without cloud base).

In [ ]:
sample_station_ids = stations["wmo_id"].head(8).tolist()
decoded, data_source = fetch_decoded_synop(sample_station_ids, START_UTC, END_UTC, stations)

print(f"Data source: {data_source}")
print(f"Rows: {len(decoded)}")
print(f"Rows with cloud base height: {decoded['cloud_base_ft'].notna().sum()}")

preview_cols = [
    "station_ID",
    "Date",
    "lat",
    "lon",
    "cloud_base_hh",
    "cloud_base_ft",
    "cloud_base_m",
    "data_source",
]
decoded[preview_cols].dropna(subset=["cloud_base_ft"]).head(20)

## 3. Download raw SYNOP bulletins for all Argentina (bulk)

Optional: inspect the raw messages behind the fallback path.

In [ ]:
raw = download_raw_synop_argentina(START_UTC, END_UTC)
print(f"{len(raw)} raw SYNOP messages")
raw.head()

## 4. Decode raw SYNOP messages (Section 3 cloud base height)

Uses the same `8NChh` parser as `fetch_decoded_synop`.

In [ ]:
decoded_from_raw = decode_raw_synop_df(raw, stations)
decoded_from_raw = normalize_observations(decoded_from_raw)
decoded_from_raw.sort_values("Date").head(20)

## 5. Quick sanity check: stations reporting cloud base height

In [ ]:
with_base = decoded.dropna(subset=["cloud_base_ft"]).copy()
print(f"Messages with cloud base height: {len(with_base)} / {len(decoded)}")

if not with_base.empty:
    display(
        with_base[
            [
                "Date",
                "station_ID",
                "station_names",
                "lat",
                "lon",
                "cloud_base_hh",
                "cloud_base_ft",
                "cloud_base_m",
                "data_source",
            ]
        ]
        .sort_values("cloud_base_ft")
        .head(15)
    )
else:
    print("No cloud base height in this time window. Try a longer range or more stations.")

## 6. Preview map (station points only)

Plots `decoded`, which always uses the normalized schema (`lat`, `lon`, `cloud_base_ft`). Interpolation over the full territory comes later.

In [ ]:
import folium

map_df = decoded.dropna(subset=["cloud_base_ft", "lat", "lon"]).copy()

if map_df.empty:
    print("No cloud-base observations to map in the selected window.")
    print(f"`decoded` has {len(decoded)} rows; data source was '{data_source}'.")
else:
    m = folium.Map(location=[-38.5, -63.5], zoom_start=4, tiles="CartoDB positron")
    for _, row in map_df.iterrows():
        hh_text = f"hh={int(row['cloud_base_hh'])}, " if pd.notna(row.get("cloud_base_hh")) else ""
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=6,
            popup=(
                f"Station {int(row['station_ID'])}<br>"
                f"UTC: {row['Date']}<br>"
                f"Cloud base: {row['cloud_base_ft']:.0f} ft ({hh_text}{row['cloud_base_m']:.0f} m)<br>"
                f"Source: {row['data_source']}"
            ),
            color="#2563eb",
            fill=True,
            fill_opacity=0.8,
        ).add_to(m)
    m

## 7. Can SYNOP be converted to AERMET?

If you meant **AERMET** (EPA's meteorological preprocessor for AERMOD):

- **Not directly.** AERMET does not ingest raw SYNOP bulletins.
- **Yes, indirectly.** After decoding SYNOP into hourly surface observations, you can export to an archive format AERMET accepts, such as **ISHD/ISD**, **SAMSON**, **CD-144**, or **EXTRACT**.
- AERMET needs more than cloud base: temperature, wind, pressure, humidity, precipitation, etc. Your decoded dataframe already contains several of these when using the OGIMET decoded path.
- For **cloud base height mapping**, you do **not** need AERMET. AERMET is mainly for air-dispersion modeling workflows.

If you meant a different format (for example an internal aviation product called AEROMET), share a sample file and we can map fields explicitly.

In [ ]:
# Optional: export the normalized observations for external tools
export_cols = [
    "Date",
    "station_ID",
    "lat",
    "lon",
    "cloud_base_hh",
    "cloud_base_ft",
    "cloud_base_m",
    "data_source",
]
decoded[export_cols].to_csv("argentina_cloud_base.csv", index=False)
print("Wrote argentina_cloud_base.csv")

## Next steps

1. **Scale up stations**: loop over all `stations['wmo_id']` in batches.
2. **Pick one timestamp**: filter `decoded` to a single UTC hour before mapping/interpolating.
3. **Interpolation**: `scipy.interpolate.griddata` or `pykrige` on a lat/lon grid clipped to Argentina's border.
4. **AERMET export** (only if needed for dispersion modeling): build an ISHD/EXTRACT writer from the decoded hourly fields.

Share this notebook from Colab via *File → Share* so coworkers can run it in their browsers.